# Train the screenjev UI detector on a GPU (Colab)

Builds the auto-labelled screenshot dataset and fine-tunes YOLO11n on it: the same pipeline as on CPU
(`bench/make_dataset.py`, `bench/train_detector.py`), just faster.

**Runtime → Change runtime type → T4 GPU** (or L4 / A100), then *Run all*.
Rough timings on a T4: dataset ~25 min (CPU-bound rendering), training 40 epochs at 960 px ~1–1.5 h.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
BRANCH = "ccr-79ef4f1c-ef404d"   # or main once merged
!git clone -q -b {BRANCH} https://github.com/NeelakandanNC/jev-experiments.git
%cd jev-experiments/02-jev-screen-control

In [ ]:
!pip install -q -r requirements.txt
!python -m playwright install --with-deps chromium > /dev/null

## 1 · Dataset
Synthetic UI pages + MiniWoB++ pages (eval-suite tasks excluded), labelled from the DOM. Same seeds as the CPU run, so the same pages.

In [ ]:
!./scripts/fetch_assets.sh
!python -m bench.make_dataset --synth 3000 --miniwob-seeds 12 --suite-seeds 4 --workers 6 --clean

In [ ]:
# eyeball a few labelled screenshots
!python -m bench.viz_labels train 6 /content/viz 1
from IPython.display import Image, display
import glob
for p in sorted(glob.glob('/content/viz/*.png'))[:6]:
    display(Image(p, width=640))

## 2 · Train

In [ ]:
!python -m bench.train_detector --device 0 --epochs 40 --imgsz 960 --batch 16 --workers 2 --cache disk --eval-imgsz 640,960,1280

In [ ]:
import json
m = json.load(open('models/screenjev-yolo.json'))
print('train hours:', m['train_hours'])
for size, res in m['metrics'].items():
    for split, r in res.items():
        print(f"{size:>10} {split:>14}  mAP50 {r['mAP50']:.3f}  mAP50-95 {r['mAP50-95']:.3f}  P {r['precision']:.3f}  R {r['recall']:.3f}")

## 3 · Get the weights back
Either download them, or push them to the branch with a GitHub token stored in Colab secrets as `GITHUB_TOKEN` (fine-grained, contents: write on this repo).

In [ ]:
from google.colab import files
files.download('models/screenjev-yolo.pt')
files.download('models/screenjev-yolo.json')

In [ ]:
# optional: commit the weights to the branch
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
!git config user.email "colab@screenjev" && git config user.name "screenjev colab"
!git add -f models/screenjev-yolo.pt models/screenjev-yolo.json
!git commit -qm "02-jev-screen-control: detector trained on GPU (Colab)"
!git push -q https://{token}@github.com/NeelakandanNC/jev-experiments.git HEAD:{BRANCH}